# NCF with request features: early and late fusion trials

Two feature-aware neural candidate generators built on top of the
`CG_ncf_with_popularity` baseline.

- **Early fusion** gives the user tower and item tower their own MLPs over
  concatenated ID embeddings and request/article features, then joins the two
  tower outputs in a final head.
- **Late fusion** first merges the user and article ID embeddings through a
  small MLP, then concatenates the request/article features before the final
  head.

Shared protocol:

- Request-level `LOSS` (BPR or BCE) from `training_negative_groups`: one clicked
  positive and one uniformly sampled observed negative per group.
- User features come from `request_profiles_<split>`: three selected interest
  category slots (zero-padded), a pooled `text64` history vector and a pooled
  `entity100` history vector.
- Article features are the article category one-hot, the article `text64`
  vector and four frozen fit-split popularity statistics.
- A standardized log1p popularity prior is blended into the score at scoring
  time. Increase `POPULARITY_WEIGHT` to make the model behave more like the
  pure popularity baseline; set it to `0` to recover the pure MLP score.
- Eligibility is `available_from <= timestamp`; every article in
  `full_history_rows` is excluded; empty-history requests fall back to
  popularity order.
- Toggle `FULL_RUN` and `TRIALS` directly in the setup code cell below.
  Smoke mode trains briefly, scores a request subset and writes subset recall.
- The first execution builds profile caches under
  `artifacts/ncf_features_cache/`; later runs reuse them. Full scoring evaluates
  the fusion MLP for every request-article pair and is the dominant runtime.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from sklearn.preprocessing import OneHotEncoder
from torch import nn

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
CG_DIR = ROOT / "5_Models" / "5_01_Candidate_Generation"
NEURAL_DIR = CG_DIR / "neural_models"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(NEURAL_DIR))
sys.path.insert(0, str(ROOT / "src"))

from cf_models import load_context, resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = ROOT / "3_Representations" / "artifacts"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
FIT_SPLIT = "train"

# Toggle the run here.
FULL_RUN = False               # False runs a two-epoch smoke pipeline.
RUN_TAG = "" if FULL_RUN else "_smoke"
TRIALS = ("early",)   # Use ("early",) or ("late",) for one trial.

if not TRIALS or set(TRIALS) - {"early", "late"}:
    raise ValueError(f"Unknown trials: {TRIALS}")

M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
EMBED_DIM = 128
TEXT_DIM = 64
ENTITY_DIM = 100
HISTORY_DIM = TEXT_DIM + ENTITY_DIM
CATEGORY_SLOTS = 3
POPULARITY_DIM = 4
POPULARITY_SMOOTHING = 20.0
POPULARITY_WEIGHT = 5.0  # Try 1, 2, 5; 0 disables the popularity blend.
POPULARITY_FEATURES = (
    "log1p_prior_exposures",
    "log1p_prior_clicks",
    "smoothed_ctr",
    "has_prior_exposure",
)
BRANCH_DIMS = (512, 256)
EARLY_HEAD_DIMS = (256, 128, 64)
LATE_ID_DIMS = (256, 128, 64)
LATE_HEAD_DIMS = (256, 128, 64, 32)
DROPOUT = 0.2
NUM_EPOCHS = 50 if FULL_RUN else 20
BATCH_SIZE = 32_768
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-6
SCORE_PAIRS = 131_072
SEED = 4222
LOSS = "bpr"  # Toggle here: "bpr" or "bce".
if LOSS not in {"bpr", "bce"}:
    raise ValueError(f"Unknown loss: {LOSS}")
PATIENCE = 5 if FULL_RUN else 3
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
SMOKE_GROUPS = 300_000
SMOKE_REQUESTS = 6_000
SMOKE_TUNE_IMPRESSIONS = 1_000
DEVICE = resolve_device("auto")
COMPARISON_PATH = RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv"

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CACHE_DIR = ARTIFACTS / "ncf_features_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Target split:", TARGET_SPLIT)
print("Trials:", ", ".join(TRIALS))
print("Mode:", "full" if FULL_RUN else "smoke")
print("Device:", DEVICE)
print("Torch:", torch.__version__)

## 2. Data ingestion and model-related processing

### Catalogue, representations and category encoder

The article index supplies the string category used by `OneHotEncoder`. The
catalogue vocabulary is the authoritative set; the three selected-interest
slots are mapped through the same vocabulary during cache construction.

In [ ]:
context = load_context(ROOT)
n_users = context.n_users
n_items = context.n_items

article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code", "category"],
).sort("article_row")
if article_index.height + 1 != n_items:
    raise ValueError("article_index does not align with context.n_items")
if not np.array_equal(
    article_index["article_row"].to_numpy(),
    np.arange(1, n_items, dtype=np.int64),
):
    raise ValueError("article_row must be contiguous from 1")
if article_index["category"].null_count():
    raise ValueError("Missing article categories")

TEXT = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
ENTITY = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")
if TEXT.shape != (n_items, TEXT_DIM):
    raise ValueError(f"Unexpected text64 shape: {TEXT.shape}")
if ENTITY.shape != (n_items, ENTITY_DIM):
    raise ValueError(f"Unexpected entity shape: {ENTITY.shape}")

article_categories = article_index["category"].to_numpy()
CATEGORY_VOCAB = sorted(set(article_categories.tolist()))
CATEGORY_DIM = len(CATEGORY_VOCAB)
CATEGORY_TO_CODE = {value: index + 1 for index, value in enumerate(CATEGORY_VOCAB)}
CATEGORY_ENCODER = OneHotEncoder(
    categories=[CATEGORY_VOCAB],
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.float32,
)
CATEGORY_ENCODER.fit(np.asarray(CATEGORY_VOCAB, dtype=object).reshape(-1, 1))
CATEGORY_LOOKUP = np.vstack(
    [
        np.zeros((1, CATEGORY_DIM), dtype=np.float32),
        CATEGORY_ENCODER.transform(
            np.asarray(CATEGORY_VOCAB, dtype=object).reshape(-1, 1)
        ).astype(np.float32),
    ]
)
CATEGORY_LOOKUP_TENSOR = torch.from_numpy(CATEGORY_LOOKUP).to(DEVICE)

article_codes = np.zeros(n_items, dtype=np.int16)
article_codes[1:] = np.asarray(
    [CATEGORY_TO_CODE[value] for value in article_categories], dtype=np.int16
)
article_onehot = CATEGORY_LOOKUP[article_codes]
encoder_check = CATEGORY_ENCODER.transform(
    np.asarray(article_categories, dtype=object).reshape(-1, 1)
)
if not np.array_equal(article_onehot[1:], encoder_check):
    raise ValueError("Category lookup does not match OneHotEncoder output")

USER_EXTRA_DIM = CATEGORY_SLOTS * CATEGORY_DIM + HISTORY_DIM
ITEM_EXTRA_DIM = CATEGORY_DIM + TEXT_DIM + POPULARITY_DIM
print(f"Users: {n_users:,}")
print(f"Articles: {n_items - 1:,}")
print(f"Categories: {CATEGORY_DIM}")
print(f"User extra dim: {USER_EXTRA_DIM}")
print(f"Item extra dim: {ITEM_EXTRA_DIM}")

### Causal request-profile caches

Each split caches one pooled 164-d history vector per request, three int16
interest-slot codes (0 = padding) and an `impression_uid -> profile_row` map.
Pooling ignores zero/missing vectors, averages the valid entries, normalises
the result and preserves repeated history references.

In [ ]:
def load_profiles(split, validate=False):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    if not parts:
        raise FileNotFoundError(f"No request profiles for {split}")
    frame = pl.read_parquet(
        parts,
        columns=[
            "impression_uid",
            "user_code",
            "timestamp",
            "full_history_rows",
            "selected_interest_categories",
        ],
    )
    if frame["impression_uid"].n_unique() != frame.height:
        raise ValueError("Duplicate impression_uid in request profiles")
    if frame["full_history_rows"].null_count():
        raise ValueError("Null full_history_rows in request profiles")
    if frame["selected_interest_categories"].null_count():
        raise ValueError("Null selected_interest_categories in request profiles")
    frame = frame.sort("timestamp", maintain_order=True)
    if validate:
        impressions = pl.read_parquet(
            PROCESSED / f"impressions_{split}.parquet",
            columns=["impression_uid", "timestamp"],
        ).with_columns(pl.col("timestamp").cast(pl.Datetime("ns")))
        check = frame.select(
            "impression_uid",
            pl.col("timestamp").cast(pl.Datetime("ns")).alias("profile_timestamp"),
        ).join(impressions, on="impression_uid", how="full", coalesce=True)
        if check.height != frame.height:
            raise ValueError("Profile/impression join changed the row count")
        if check["profile_timestamp"].null_count() or check["timestamp"].null_count():
            raise ValueError("Missing profile or impression timestamp")
        if not (check["profile_timestamp"] == check["timestamp"]).all():
            raise ValueError("Profile timestamps do not match impressions")
    return frame

In [ ]:
def pool_profiles(history_rows, matrix, block=2_000):
    matrix = np.asarray(matrix)
    if matrix.ndim != 2:
        raise ValueError("matrix must be two-dimensional")
    n = len(history_rows)
    dim = matrix.shape[1]
    result = np.zeros((n, dim), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        total = int(lengths.sum())
        if total == 0:
            continue
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        if flat.min() < 0 or flat.max() >= matrix.shape[0]:
            raise ValueError("history contains an invalid article row")
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12
        if not valid.any():
            continue

        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        valid_groups = groups[valid]
        valid_vectors = vectors[valid]
        starts = np.r_[0, np.flatnonzero(valid_groups[1:] != valid_groups[:-1]) + 1]
        sums = np.add.reduceat(valid_vectors, starts, axis=0)
        counts = np.diff(np.r_[starts, len(valid_groups)])
        pooled = sums / counts[:, None]
        norms = np.linalg.norm(pooled, axis=1)
        nonzero = norms > 0
        pooled[nonzero] /= norms[nonzero, None]
        result[start : start + len(chunk)][valid_groups[starts]] = pooled
    return result

In [ ]:
def pool_history(history_rows, text, entity, block=2_000):
    n = len(history_rows)
    out = np.zeros((n, HISTORY_DIM), dtype=np.float32)
    text_profile = pool_profiles(history_rows, text, block=block)
    entity_profile = pool_profiles(history_rows, entity, block=block)
    out[:, :TEXT_DIM] = text_profile
    out[:, TEXT_DIM:] = entity_profile
    return out

In [ ]:
def encode_interest_slots(category_lists, strict):
    codes = np.zeros((len(category_lists), CATEGORY_SLOTS), dtype=np.int16)
    unknown = 0
    for row, values in enumerate(category_lists):
        for slot, value in enumerate(values[:CATEGORY_SLOTS]):
            code = CATEGORY_TO_CODE.get(value)
            if code is None:
                unknown += 1
                if strict:
                    raise ValueError(f"Unknown interest category: {value!r}")
                continue
            codes[row, slot] = code
    return codes, unknown

In [ ]:
def profile_cache_paths(split):
    base = CACHE_DIR / split
    return {
        "features": base / "profile_features.npy",
        "slots": base / "interest_codes.npy",
        "uids": base / "profile_uids.npy",
        "users": base / "profile_users.npy",
        "index": base / "profile_index.parquet",
        "meta": base / "cache_meta.json",
    }

In [ ]:
def build_profile_cache(split, profiles):
    paths = profile_cache_paths(split)
    base = paths["features"].parent
    base.mkdir(parents=True, exist_ok=True)
    n = profiles.height
    features = np.lib.format.open_memmap(
        paths["features"],
        mode="w+",
        dtype=np.float32,
        shape=(n, HISTORY_DIM),
    )
    slots = np.empty((n, CATEGORY_SLOTS), dtype=np.int16)
    unknown = 0
    for start in range(0, n, 20_000):
        end = min(start + 20_000, n)
        rows = profiles["full_history_rows"].slice(start, end - start).to_list()
        features[start:end] = pool_history(rows, TEXT, ENTITY)
        categories = (
            profiles["selected_interest_categories"]
            .slice(start, end - start)
            .to_list()
        )
        codes, count = encode_interest_slots(
            categories, strict=(split == FIT_SPLIT)
        )
        slots[start:end] = codes
        unknown += count
    features.flush()

    uids = profiles["impression_uid"].to_numpy().astype(np.int64)
    users = profiles["user_code"].to_numpy().astype(np.int64)
    np.save(paths["slots"], slots)
    np.save(paths["uids"], uids)
    np.save(paths["users"], users)
    pl.DataFrame(
        {
            "impression_uid": uids,
            "profile_row": np.arange(n, dtype=np.int32),
        }
    ).write_parquet(paths["index"])
    meta = {
        "split": split,
        "requests": int(n),
        "history_dim": HISTORY_DIM,
        "text_dim": TEXT_DIM,
        "entity_dim": ENTITY_DIM,
        "category_slots": CATEGORY_SLOTS,
        "category_dim": CATEGORY_DIM,
        "category_vocab": CATEGORY_VOCAB,
        "unknown_interest_values": int(unknown),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    paths["meta"].write_text(json.dumps(meta, indent=2) + "\n")
    print(f"Built profile cache for {split}: {n:,} requests")
    return {
        "features": np.load(paths["features"], mmap_mode="r"),
        "slots": np.load(paths["slots"], mmap_mode="r"),
        "uids": uids,
        "users": users,
        "index": pl.read_parquet(paths["index"]),
        "meta": meta,
    }

In [ ]:
def load_profile_cache(split, validate=False):
    profiles = load_profiles(split, validate=validate)
    paths = profile_cache_paths(split)
    ready = all(paths[name].exists() for name in paths)
    if ready:
        try:
            meta = json.loads(paths["meta"].read_text())
        except (OSError, json.JSONDecodeError):
            meta = {}
        ready = (
            meta.get("category_vocab") == CATEGORY_VOCAB
            and meta.get("history_dim") == HISTORY_DIM
            and meta.get("category_slots") == CATEGORY_SLOTS
            and meta.get("requests") == profiles.height
        )
    if ready:
        features = np.load(paths["features"], mmap_mode="r")
        slots = np.load(paths["slots"], mmap_mode="r")
        uids = np.load(paths["uids"])
        users = np.load(paths["users"])
        index = pl.read_parquet(paths["index"])
        ready = (
            features.shape == (profiles.height, HISTORY_DIM)
            and slots.shape == (profiles.height, CATEGORY_SLOTS)
            and len(uids) == profiles.height
            and len(users) == profiles.height
            and np.array_equal(uids, profiles["impression_uid"].to_numpy())
            and np.array_equal(index["impression_uid"].to_numpy(), uids)
        )
    if not ready:
        return profiles, build_profile_cache(split, profiles)
    return profiles, {
        "features": features,
        "slots": slots,
        "uids": uids,
        "users": users,
        "index": index,
        "meta": meta,
    }

In [ ]:
train_profiles, train_cache = load_profile_cache(FIT_SPLIT, validate=False)

if TARGET_SPLIT == FIT_SPLIT:
    target_profiles, target_cache = train_profiles, train_cache
else:
    target_profiles, target_cache = load_profile_cache(TARGET_SPLIT, validate=True)
    train_profiles = train_profiles.select(
        "impression_uid", "user_code", "timestamp"
    )

if not np.array_equal(
    target_cache["uids"], target_profiles["impression_uid"].to_numpy()
):
    raise ValueError("Target profile cache does not align with request profiles")

print(f"Training requests: {train_profiles.height:,}")

print(f"Target requests: {target_profiles.height:,}")

### Article features and the time-ordered catalogue

The item extra block is static: category one-hot, `text64` and the four
popularity statistics. The catalogue is sorted by `available_from` so each
request timestamp selects a prefix.

In [ ]:
counts = pl.read_parquet(
    PROCESSED / "news_catalogue.parquet",
    columns=[
        "news_code",
        "exposures_train",
        "clicks_train",
        "exposures_validation",
        "clicks_validation",
    ],
)
prior_exposures = pl.col("exposures_train")
prior_clicks = pl.col("clicks_train")
if TARGET_SPLIT != "validation":
    prior_exposures = prior_exposures + pl.col("exposures_validation")
    prior_clicks = prior_clicks + pl.col("clicks_validation")
counts = counts.select(
    "news_code",
    prior_exposures.alias("prior_exposures"),
    prior_clicks.alias("prior_clicks"),
)
joined_counts = (
    article_index.select("article_row", "news_code")
    .join(counts, on="news_code", how="left")
    .sort("article_row")
)
if joined_counts["prior_exposures"].null_count():
    raise ValueError("Missing prior_exposures for some articles")
if joined_counts["prior_clicks"].null_count():
    raise ValueError("Missing prior_clicks for some articles")

pop_rows = joined_counts["article_row"].to_numpy().astype(np.int64)
pop_exposures = joined_counts["prior_exposures"].to_numpy().astype(np.float32)
pop_clicks = joined_counts["prior_clicks"].to_numpy().astype(np.float32)
POPULARITY_TABLE = np.zeros((n_items, POPULARITY_DIM), dtype=np.float32)
POPULARITY_TABLE[pop_rows, 0] = np.log1p(pop_exposures)
POPULARITY_TABLE[pop_rows, 1] = np.log1p(pop_clicks)
POPULARITY_TABLE[pop_rows, 2] = np.where(
    pop_exposures > 0,
    (pop_clicks + 1.0) / (pop_exposures + POPULARITY_SMOOTHING),
    0.0,
).astype(np.float32)
POPULARITY_TABLE[pop_rows, 3] = (pop_exposures > 0).astype(np.float32)

prior_log = POPULARITY_TABLE[:, 0]
prior_mask = pop_exposures > 0
if int(prior_mask.sum()) > 1:
    POPULARITY_PRIOR_MEAN = float(prior_log[pop_rows[prior_mask]].mean())
    POPULARITY_PRIOR_STD = float(prior_log[pop_rows[prior_mask]].std())
else:
    POPULARITY_PRIOR_MEAN = 0.0
    POPULARITY_PRIOR_STD = 1.0
POPULARITY_PRIOR = (
    (prior_log - POPULARITY_PRIOR_MEAN)
    / max(POPULARITY_PRIOR_STD, 1e-6)
).astype(np.float32)
POPULARITY_PRIOR[0] = 0.0

TEXT_ALL = np.asarray(TEXT, dtype=np.float32)
ITEM_EXTRA = np.concatenate(
    [article_onehot.astype(np.float32), TEXT_ALL, POPULARITY_TABLE],
    axis=1,
).astype(np.float32)
if ITEM_EXTRA.shape != (n_items, ITEM_EXTRA_DIM):
    raise ValueError(f"Unexpected item extra shape: {ITEM_EXTRA.shape}")
if not np.isfinite(ITEM_EXTRA).all():
    raise ValueError("Item extra contains non-finite values")
ITEM_EXTRA_TENSOR = torch.from_numpy(ITEM_EXTRA).to(DEVICE)

catalogue = pl.DataFrame(
    {
        "article_row": np.arange(1, n_items, dtype=np.int32),
        "news_code": context.news_code[1:].astype(np.int32),
        "available_from": context.available_from[1:],
        "popularity": context.popularity[1:].astype(np.float32),
    }
)
ordered = catalogue.sort(["available_from", "news_code"])
ARTICLE_ROWS_SORTED = ordered["article_row"].to_numpy().astype(np.int64)
NEWS_SORTED = ordered["news_code"].to_numpy().astype(np.int64)
AVAILABLE_SORTED = ordered["available_from"].to_numpy().astype("datetime64[ns]")
POPULARITY_SORTED = ordered["popularity"].to_numpy().astype(np.float32)
POPULARITY_PRIOR_SORTED = POPULARITY_PRIOR[ARTICLE_ROWS_SORTED]
POS_IN_SORTED = np.full(n_items, -1, dtype=np.int64)
POS_IN_SORTED[ARTICLE_ROWS_SORTED] = np.arange(
    ARTICLE_ROWS_SORTED.size, dtype=np.int64
)
GLOBAL_POPULARITY_ORDER = np.lexsort((NEWS_SORTED, -POPULARITY_SORTED))
if not np.all(AVAILABLE_SORTED[1:] >= AVAILABLE_SORTED[:-1]):
    raise ValueError("Availability order is not non-decreasing")
print("Item extra:", ITEM_EXTRA.shape)
print("Catalogue:", len(ARTICLE_ROWS_SORTED))

### Request-level BPR training data

`training_negative_groups` contains each clicked positive and the observed
non-clicked articles from the same slate. One negative is sampled per positive
group in each epoch.

In [ ]:
def load_training_groups():
    parts = sorted((FEATURES / "training_negative_groups").glob("*.parquet"))
    if not parts:
        raise FileNotFoundError("No training_negative_groups partitions")
    groups = pl.read_parquet(parts)
    if groups["negative_news_codes"].list.len().min() < 1:
        raise ValueError("A training group has no negative article")
    if not FULL_RUN and groups.height > SMOKE_GROUPS:
        groups = groups.sample(n=SMOKE_GROUPS, seed=SEED, shuffle=True)
    return groups

In [ ]:
training_groups = load_training_groups().with_row_index("group_row")

positive_lookup = article_index.select(
    pl.col("news_code"),
    pl.col("article_row").alias("pos_row"),
)

training_groups = training_groups.join(
    positive_lookup,
    left_on="positive_news_code",
    right_on="news_code",
    how="left",
)

if training_groups["pos_row"].null_count():
    raise ValueError("A positive article is missing from article_index")

negative_lookup = article_index.select(
    pl.col("news_code").alias("negative_news_code"),
    pl.col("article_row").alias("neg_row"),
)

negative_long = (
    training_groups.select("group_row", "negative_news_codes")
    .explode("negative_news_codes")
    .rename({"negative_news_codes": "negative_news_code"})
    .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
    .join(negative_lookup, on="negative_news_code", how="left")
    .sort(["group_row", "neg_pos"])
)

if negative_long["neg_row"].null_count():
    raise ValueError("A negative article is missing from article_index")

n_groups = training_groups.height

NEG_MAX = int(training_groups["negative_news_codes"].list.len().max())

NEG_MATRIX = np.zeros((n_groups, NEG_MAX), dtype=np.int64)

group_rows = negative_long["group_row"].to_numpy().astype(np.int64)

negative_positions = negative_long["neg_pos"].to_numpy().astype(np.int64)

negative_rows = negative_long["neg_row"].to_numpy().astype(np.int64)

NEG_MATRIX[group_rows, negative_positions] = negative_rows

NEG_LEN = np.bincount(group_rows, minlength=n_groups).astype(np.int64)

if (NEG_LEN < 1).any():
    raise ValueError("At least one training group has no negative")

training_groups = (
    training_groups.join(train_cache["index"], on="impression_uid", how="left")
    .join(
        train_profiles.select("impression_uid", "timestamp"),
        on="impression_uid",
        how="left",
    )
    .sort("group_row")
)

if training_groups["profile_row"].null_count():
    raise ValueError("Missing request profile for a training group")

if not np.array_equal(
    training_groups["group_row"].to_numpy().astype(np.int64),
    np.arange(n_groups, dtype=np.int64),
):
    raise ValueError("Training group order changed during joins")

TRAIN_PROFILE_IDX = training_groups["profile_row"].to_numpy().astype(np.int64)

TRAIN_USER_IDS = train_cache["users"][TRAIN_PROFILE_IDX]

TRAIN_POS_ROWS = training_groups["pos_row"].to_numpy().astype(np.int64)

if TRAIN_PROFILE_IDX.max() >= train_profiles.height:
    raise ValueError("Training profile row is outside the cache")

if TRAIN_POS_ROWS.min() < 1 or TRAIN_POS_ROWS.max() >= n_items:
    raise ValueError("Training positive article row is outside the catalogue")

if negative_rows.min() < 1 or negative_rows.max() >= n_items:
    raise ValueError("Training negative article row is outside the catalogue")

print(f"Training groups: {n_groups:,}")

print(f"Max observed negatives per group: {NEG_MAX}")

TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

cutoff_ns = chronological_cutoff(
    training_groups["timestamp"].to_numpy(), TUNE_QUANTILE
)

fit_mask = chronological_mask(
    training_groups["timestamp"].to_numpy(), cutoff_ns
)

fit_idx = np.flatnonzero(fit_mask)

FIT_PROFILE_IDX = TRAIN_PROFILE_IDX[fit_idx]

FIT_USER_IDS = TRAIN_USER_IDS[fit_idx]

FIT_POS_ROWS = TRAIN_POS_ROWS[fit_idx]

FIT_NEG_MATRIX = NEG_MATRIX[fit_idx]

FIT_NEG_LEN = NEG_LEN[fit_idx]

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    train_cache["index"],
    max_impressions=None if FULL_RUN else SMOKE_TUNE_IMPRESSIONS,
    seed=SEED + 1,
)

if tune["article_row"].null_count() or tune["profile_row"].null_count():
    raise ValueError("Tuning rows have a missing article or profile")

TUNE_PROFILE_IDX = tune["profile_row"].to_numpy().astype(np.int64)

TUNE_USER_IDS = train_cache["users"][TUNE_PROFILE_IDX]

TUNE_ITEM_ROWS = tune["article_row"].to_numpy().astype(np.int64)

TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)

TUNE_UIDS = tune["impression_uid"].to_numpy()

print(f"Chronological fit groups: {len(fit_idx):,} / {n_groups:,}")

print(f"Tuning cutoff: {np.datetime64(cutoff_ns, 'ns')}")

print(f"Tuning rows: {tune.height:,}")

print(
    "Tuning NDCG k (max training clicks per impression): "
    f"{TUNE_NDCG_K}"
)

## 3. Setting up the model

### Fusion models

`EarlyFusionNCF` has separate user and item towers and a final head.
`LateFusionNCF` fuses the two ID embeddings first and appends the extra
features only in the final head. Both models apply dropout after hidden
layers and at the input/fusion boundaries.

In [ ]:
def mlp_stack(input_dim, hidden_dims, dropout=DROPOUT):
    layers = []
    dim = input_dim
    for hidden_dim in hidden_dims:
        layers.append(nn.Linear(dim, hidden_dim))
        layers.append(nn.GELU())
        layers.append(nn.Dropout(dropout))
        dim = hidden_dim
    return nn.Sequential(*layers)

In [ ]:
def mlp_head(input_dim, hidden_dims, dropout=DROPOUT):
    layers = []
    dim = input_dim
    for hidden_dim in hidden_dims:
        layers.append(nn.Linear(dim, hidden_dim))
        layers.append(nn.GELU())
        layers.append(nn.Dropout(dropout))
        dim = hidden_dim
    layers.append(nn.Linear(dim, 1))
    return nn.Sequential(*layers)

In [ ]:
class EarlyFusionNCF(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        embed_dim,
        user_extra_dim,
        item_extra_dim,
        branch_dims,
        head_dims,
        dropout,
    ):
        super().__init__()
        self.branch_dim = branch_dims[-1]
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()
        self.input_dropout = nn.Dropout(dropout)
        self.fusion_dropout = nn.Dropout(dropout)
        self.user_tower = mlp_stack(
            embed_dim + user_extra_dim, branch_dims, dropout
        )
        self.item_tower = mlp_stack(
            embed_dim + item_extra_dim, branch_dims, dropout
        )
        self.head = mlp_head(2 * self.branch_dim, head_dims, dropout)

    def encode_users(self, user_ids, user_extra):
        return self.user_tower(
            self.input_dropout(
                torch.cat(
                    [self.user_embedding(user_ids), user_extra], dim=-1
                )
            )
        )

    def encode_items(self, item_ids, item_extra):
        return self.item_tower(
            self.input_dropout(
                torch.cat(
                    [self.item_embedding(item_ids), item_extra], dim=-1
                )
            )
        )

    def score_branches(self, user_branch, item_branch):
        return self.head(
            self.fusion_dropout(
                torch.cat([user_branch, item_branch], dim=-1)
            )
        ).squeeze(-1)

    def score_pairs(self, user_ids, item_ids, user_extra, item_extra):
        return self.score_branches(
            self.encode_users(user_ids, user_extra),
            self.encode_items(item_ids, item_extra),
        )

In [ ]:
class LateFusionNCF(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        embed_dim,
        user_extra_dim,
        item_extra_dim,
        id_dims,
        head_dims,
        dropout,
    ):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()
        self.input_dropout = nn.Dropout(dropout)
        self.fusion_dropout = nn.Dropout(dropout)
        self.id_fusion = mlp_stack(2 * embed_dim, id_dims, dropout)
        self.head = mlp_head(
            id_dims[-1] + user_extra_dim + item_extra_dim,
            head_dims,
            dropout,
        )

    def fuse_embeddings(self, user_emb, item_emb):
        return self.id_fusion(
            self.input_dropout(torch.cat([user_emb, item_emb], dim=-1))
        )

    def score_fused(self, fused, user_extra, item_extra):
        return self.head(
            self.fusion_dropout(
                torch.cat([fused, user_extra, item_extra], dim=-1)
            )
        ).squeeze(-1)

    def score_pairs(self, user_ids, item_ids, user_extra, item_extra):
        return self.score_fused(
            self.fuse_embeddings(
                self.user_embedding(user_ids),
                self.item_embedding(item_ids),
            ),
            user_extra,
            item_extra,
        )

In [ ]:
def build_model(trial):
    if trial == "early":
        return EarlyFusionNCF(
            n_users,
            n_items,
            EMBED_DIM,
            USER_EXTRA_DIM,
            ITEM_EXTRA_DIM,
            BRANCH_DIMS,
            EARLY_HEAD_DIMS,
            DROPOUT,
        ).to(DEVICE)
    if trial == "late":
        return LateFusionNCF(
            n_users,
            n_items,
            EMBED_DIM,
            USER_EXTRA_DIM,
            ITEM_EXTRA_DIM,
            LATE_ID_DIMS,
            LATE_HEAD_DIMS,
            DROPOUT,
        ).to(DEVICE)
    raise ValueError(f"Unknown trial: {trial}")

In [ ]:
def build_user_extra(slots, history):
    slot_tensor = torch.as_tensor(
        np.asarray(slots), dtype=torch.long, device=DEVICE
    )
    onehot = CATEGORY_LOOKUP_TENSOR[slot_tensor].reshape(len(slots), -1)
    history_tensor = torch.as_tensor(
        np.asarray(history), dtype=torch.float32, device=DEVICE
    )
    return torch.cat([onehot, history_tensor], dim=1)

## 4. Training the model

### Training functions

Early fusion precomputes the item tower once because article features are
static. Late fusion evaluates the pair-level ID-fusion layers in bounded
chunks. Both use the same request-level `LOSS` objective (BPR or BCE).

In [ ]:
def train_epoch(
    model,
    optimizer,
    user_ids,
    profile_idx,
    pos_rows,
    neg_matrix,
    neg_len,
    user_features,
    user_slots,
    item_extra_tensor,
    rng,
):
    model.train()
    order = rng.permutation(len(user_ids))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        choice = rng.integers(0, neg_len[batch])
        neg_rows = neg_matrix[batch, choice]
        profile_rows = profile_idx[batch]
        users = torch.as_tensor(
            user_ids[batch], dtype=torch.long, device=DEVICE
        )
        user_extra = build_user_extra(
            user_slots[profile_rows], user_features[profile_rows]
        )
        positives = torch.as_tensor(
            pos_rows[batch], dtype=torch.long, device=DEVICE
        )
        negatives = torch.as_tensor(
            neg_rows, dtype=torch.long, device=DEVICE
        )
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model.score_pairs(
            users, positives, user_extra, item_extra_tensor[positives]
        )
        negative_scores = model.score_pairs(
            users, negatives, user_extra, item_extra_tensor[negatives]
        )
        if LOSS == "bpr":
            loss = -F.logsigmoid(positive_scores - negative_scores).mean()
        else:
            scores = torch.cat([positive_scores, negative_scores])
            labels = torch.cat(
                [
                    torch.ones_like(positive_scores),
                    torch.zeros_like(negative_scores),
                ]
            )
            loss = F.binary_cross_entropy_with_logits(scores, labels)
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


In [ ]:
def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), SCORE_PAIRS):
            end = min(start + SCORE_PAIRS, len(TUNE_CLICKED))
            profile_idx = TUNE_PROFILE_IDX[start:end]
            users = torch.as_tensor(
                TUNE_USER_IDS[start:end], dtype=torch.long, device=DEVICE
            )
            items = torch.as_tensor(
                TUNE_ITEM_ROWS[start:end], dtype=torch.long, device=DEVICE
            )
            user_extra = build_user_extra(
                train_cache["slots"][profile_idx],
                train_cache["features"][profile_idx],
            )
            scores[start:end] = model.score_pairs(
                users,
                items,
                user_extra,
                ITEM_EXTRA_TENSOR[items],
            ).cpu().numpy()
    _, ndcg = mean_slate_metrics(
        TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K
    )
    return ndcg

In [ ]:
def train_model(model, trial):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 1)
    training_log = []
    stopper = EarlyStopper(patience=PATIENCE)
    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        loss = train_epoch(
            model,
            optimizer,
            FIT_USER_IDS,
            FIT_PROFILE_IDX,
            FIT_POS_ROWS,
            FIT_NEG_MATRIX,
            FIT_NEG_LEN,
            train_cache["features"],
            train_cache["slots"],
            ITEM_EXTRA_TENSOR,
            rng,
        )
        ndcg = tune_ndcg(model)
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "tune",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": ndcg,
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"[{trial}] epoch {epoch:02d} loss={loss:.6f} "
            f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
        )
        if stopper.step(ndcg, model, epoch):
            print(f"[{trial}] Early stopping")
            break

    stopper.restore(model)
    print(
        f"[{trial}] Best epoch: {stopper.best_epoch} "
        f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
    )

    final_model = build_model(trial)
    optimizer = torch.optim.AdamW(
        final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 2)
    for epoch in range(1, stopper.best_epoch + 1):
        started = time.time()
        loss = train_epoch(
            final_model,
            optimizer,
            TRAIN_USER_IDS,
            TRAIN_PROFILE_IDX,
            TRAIN_POS_ROWS,
            NEG_MATRIX,
            NEG_LEN,
            train_cache["features"],
            train_cache["slots"],
            ITEM_EXTRA_TENSOR,
            rng,
        )
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "refit",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": float("nan"),
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"[{trial}] refit epoch {epoch:02d} "
            f"loss={loss:.6f} ({elapsed:.1f}s)"
        )
    return final_model, training_log, stopper

In [ ]:
def save_model_artifact(trial, model, training_log, stopper, artifact_dir):
    model_name = f"ncf_features_{trial}"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "category_vocab": CATEGORY_VOCAB,
            "popularity_table": torch.from_numpy(POPULARITY_TABLE),
            "popularity_prior": torch.from_numpy(POPULARITY_PRIOR),
            "config": {
                "trial": trial,
                "model": model_name,
                "n_users": n_users,
                "n_items": n_items,
                "embed_dim": EMBED_DIM,
                "user_extra_dim": USER_EXTRA_DIM,
                "item_extra_dim": ITEM_EXTRA_DIM,
                "category_dim": CATEGORY_DIM,
                "category_slots": CATEGORY_SLOTS,
                "popularity_weight": POPULARITY_WEIGHT,
                "popularity_prior_mean": POPULARITY_PRIOR_MEAN,
                "popularity_prior_std": POPULARITY_PRIOR_STD,
                "history_dim": HISTORY_DIM,
                "branch_dims": list(BRANCH_DIMS),
                "early_head_dims": list(EARLY_HEAD_DIMS),
                "late_id_dims": list(LATE_ID_DIMS),
                "late_head_dims": list(LATE_HEAD_DIMS),
                "dropout": DROPOUT,
                "max_epochs": NUM_EPOCHS,
                "best_epoch": int(stopper.best_epoch),
                "epochs": int(stopper.best_epoch),
                "tune_ndcg": float(stopper.best_score),
                "ndcg_k": int(TUNE_NDCG_K),
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "seed": SEED,
                "objective": LOSS,
            },
        },
        artifact_dir / "model.pt",
    )
    pl.DataFrame(training_log).write_csv(artifact_dir / "training_log.csv")
    print(f"[{trial}] Saved model:", artifact_dir / "model.pt")

In [ ]:
TRAINED_TRIALS = {}
for trial in TRIALS:
    model_name = f"ncf_features_{trial}"
    artifact_dir = ARTIFACTS / f"{model_name}{RUN_TAG}"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    model = build_model(trial)
    print(f"[{trial}] Training {model_name}")
    model, training_log, stopper = train_model(model, trial)
    save_model_artifact(trial, model, training_log, stopper, artifact_dir)
    TRAINED_TRIALS[trial] = {
        "model": model,
        "model_name": model_name,
        "artifact_dir": artifact_dir,
        "training_log": training_log,
        "stopper": stopper,
    }

## 5. Results

### Candidate selection helpers

These mirror the shared candidate wire format and deterministic tie-breaking
used by the other candidate generators.

In [ ]:
def popularity_select(order, n_eligible, excluded=None, k=CANDIDATE_CAP):
    excluded_set = set() if excluded is None else set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position >= n_eligible or position in excluded_set:
            continue
        selected.append(position)
        if len(selected) == k:
            return np.asarray(selected, dtype=np.int64)
    raise ValueError(
        f"only {len(selected)} popular eligible articles; cannot select {k}"
    )

In [ ]:
def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    if pool.size < count:
        raise ValueError(
            f"only {pool.size} eligible articles remain; cannot fill {count}"
        )
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]

In [ ]:
def select_top_k(
    scores, popularity, news_code, k, excluded=None, fallback_order=None
):
    scores = np.asarray(scores, dtype=np.float32)
    popularity = np.asarray(popularity)
    news_code = np.asarray(news_code)
    if scores.ndim != 1:
        raise ValueError("scores must be one-dimensional")
    if popularity.shape != scores.shape or news_code.shape != scores.shape:
        raise ValueError("score, popularity and news_code shapes must match")
    if k < 1:
        raise ValueError("k must be positive")
    n = scores.size
    if n < k:
        raise ValueError(f"only {n} eligible articles; cannot select {k}")
    if fallback_order is not None:
        return popularity_select(fallback_order, n, excluded=excluded, k=k)

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        if positions.min() < 0 or positions.max() >= n:
            raise ValueError("excluded position outside the eligible array")
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            tie_order = np.lexsort((news_code[ties], -popularity[ties]))
            ties = ties[tie_order[:need]]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        fill = popularity_fill(
            remaining, popularity, news_code, k - selected.size
        )
        selected = np.r_[selected, fill]

    return selected

### Scoring, manifest and evaluation

In [ ]:
def encode_early_items(model, item_extra_tensor, batch=65_536):
    out = np.empty((n_items, model.branch_dim), dtype=np.float32)
    ids = torch.arange(n_items, dtype=torch.long, device=DEVICE)
    model.eval()
    with torch.no_grad():
        for start in range(0, n_items, batch):
            end = min(start + batch, n_items)
            out[start:end] = model.encode_items(
                ids[start:end], item_extra_tensor[start:end]
            ).cpu().numpy()
    return torch.from_numpy(out).to(DEVICE)

In [ ]:
def score_early(model, user_branch, item_branch, max_pairs=SCORE_PAIRS):
    n_users = user_branch.shape[0]
    n_items_local = item_branch.shape[0]
    scores = np.empty((n_users, n_items_local), dtype=np.float32)
    if n_users == 0 or n_items_local == 0:
        return scores
    items_per_chunk = min(n_items_local, max_pairs)
    users_per_chunk = max(1, max_pairs // items_per_chunk)
    with torch.no_grad():
        for u0 in range(0, n_users, users_per_chunk):
            u1 = min(u0 + users_per_chunk, n_users)
            user_block = user_branch[u0:u1]
            for i0 in range(0, n_items_local, items_per_chunk):
                i1 = min(i0 + items_per_chunk, n_items_local)
                item_block = item_branch[i0:i1]
                rows = u1 - u0
                cols = i1 - i0
                user_expand = user_block[:, None, :].expand(
                    rows, cols, model.branch_dim
                )
                item_expand = item_block[None, :, :].expand(
                    rows, cols, model.branch_dim
                )
                scores[u0:u1, i0:i1] = (
                    model.score_branches(user_expand, item_expand)
                    .cpu()
                    .numpy()
                )
    return scores

In [ ]:
def score_late(
    model,
    user_ids,
    user_extra,
    item_emb,
    item_extra,
    max_pairs=SCORE_PAIRS,
):
    n_users = len(user_ids)
    n_items_local = item_emb.shape[0]
    scores = np.empty((n_users, n_items_local), dtype=np.float32)
    if n_users == 0 or n_items_local == 0:
        return scores
    user_emb = model.user_embedding(
        torch.as_tensor(user_ids, dtype=torch.long, device=DEVICE)
    )
    items_per_chunk = min(n_items_local, max_pairs)
    users_per_chunk = max(1, max_pairs // items_per_chunk)
    with torch.no_grad():
        for u0 in range(0, n_users, users_per_chunk):
            u1 = min(u0 + users_per_chunk, n_users)
            for i0 in range(0, n_items_local, items_per_chunk):
                i1 = min(i0 + items_per_chunk, n_items_local)
                rows = u1 - u0
                cols = i1 - i0
                fused = model.fuse_embeddings(
                    user_emb[u0:u1, None, :].expand(
                        rows, cols, EMBED_DIM
                    ),
                    item_emb[None, i0:i1, :].expand(
                        rows, cols, EMBED_DIM
                    ),
                )
                scores[u0:u1, i0:i1] = model.score_fused(
                    fused,
                    user_extra[u0:u1, None, :].expand(
                        rows, cols, USER_EXTRA_DIM
                    ),
                    item_extra[None, i0:i1, :].expand(
                        rows, cols, ITEM_EXTRA_DIM
                    ),
                ).cpu().numpy()
    return scores

In [ ]:
def score_target_split(trial, model, artifact_dir):
    recommendations_path = (
        artifact_dir / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"
    )
    timestamps = target_profiles["timestamp"].to_numpy().astype("datetime64[ns]")
    history_rows = target_profiles["full_history_rows"].to_list()
    has_history = np.fromiter(
        (len(rows) > 0 for rows in history_rows),
        dtype=bool,
        count=len(history_rows),
    )
    impression_uids = target_cache["uids"]

    starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
    ends = np.r_[starts[1:], len(timestamps)]
    batches = list(zip(starts, ends))
    if not FULL_RUN:
        batch_rng = np.random.default_rng(SEED)
        batch_order = batch_rng.permutation(len(batches))
        selected_batches = []
        total = 0
        for index in batch_order:
            start, end = batches[index]
            selected_batches.append((start, end))
            total += end - start
            if total >= SMOKE_REQUESTS:
                break
        batches = sorted(selected_batches)
    processed_requests = int(sum(end - start for start, end in batches))
    print(f"[{trial}] Scoring requests: {processed_requests:,} / {len(timestamps):,}")

    model.eval()
    item_branch_sorted = None
    item_emb_sorted = None
    item_extra_sorted = None
    sorted_ids = torch.as_tensor(
        ARTICLE_ROWS_SORTED, dtype=torch.long, device=DEVICE
    )
    if trial == "early":
        item_branch_all = encode_early_items(model, ITEM_EXTRA_TENSOR)
        item_branch_sorted = item_branch_all.index_select(0, sorted_ids)
    else:
        with torch.no_grad():
            item_emb_sorted = model.item_embedding(sorted_ids)
            item_extra_sorted = ITEM_EXTRA_TENSOR[sorted_ids]

    cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
    writer = CandidateWriter(recommendations_path)
    processed_uids = []
    started = time.time()
    try:
        for batch_no, (start, end) in enumerate(batches, 1):
            timestamp = timestamps[start]
            n_eligible = int(
                np.searchsorted(AVAILABLE_SORTED, timestamp, side="right")
            )
            if n_eligible < CANDIDATE_CAP:
                raise ValueError(
                    f"Only {n_eligible} eligible articles at {timestamp}"
                )

            user_ids = target_cache["users"][start:end]
            user_extra = build_user_extra(
                target_cache["slots"][start:end],
                target_cache["features"][start:end],
            )
            if trial == "early":
                with torch.no_grad():
                    user_branch = model.encode_users(
                        torch.as_tensor(
                            user_ids, dtype=torch.long, device=DEVICE
                        ),
                        user_extra,
                    )
                scores = score_early(
                    model, user_branch, item_branch_sorted[:n_eligible]
                )
            else:
                scores = score_late(
                    model,
                    user_ids,
                    user_extra,
                    item_emb_sorted[:n_eligible],
                    item_extra_sorted[:n_eligible],
                )

            scores = scores + (
                POPULARITY_WEIGHT
                * POPULARITY_PRIOR_SORTED[:n_eligible][None, :]
            )

            popularity_batch = POPULARITY_SORTED[:n_eligible]
            news_batch = NEWS_SORTED[:n_eligible]
            selected_batch = np.empty(
                (end - start, CANDIDATE_CAP), dtype=np.int64
            )
            for local, profile_row in enumerate(range(start, end)):
                rows = history_rows[profile_row]
                if rows:
                    excluded = POS_IN_SORTED[
                        np.asarray(rows, dtype=np.int64)
                    ]
                    excluded = excluded[
                        (excluded >= 0) & (excluded < n_eligible)
                    ]
                else:
                    excluded = np.empty(0, dtype=np.int64)
                fallback = (
                    None
                    if has_history[profile_row]
                    else GLOBAL_POPULARITY_ORDER
                )
                selected_batch[local] = select_top_k(
                    scores[local],
                    popularity_batch,
                    news_batch,
                    CANDIDATE_CAP,
                    excluded=excluded,
                    fallback_order=fallback,
                )

            processed_uids.append(impression_uids[start:end].copy())
            writer.add(
                np.repeat(impression_uids[start:end], CANDIDATE_CAP),
                np.tile(cutoff_values, end - start),
                news_batch[selected_batch.reshape(-1)],
            )
            if batch_no % 1000 == 0 or batch_no == len(batches):
                print(
                    f"[{trial}] Scored {batch_no:,}/{len(batches):,} "
                    f"timestamp batches; {writer.rows:,} rows",
                    flush=True,
                )
    finally:
        writer.close()

    elapsed = time.time() - started
    expected_rows = processed_requests * CANDIDATE_CAP
    actual_rows = (
        pl.scan_parquet(recommendations_path)
        .select(pl.len())
        .collect()
        .item()
    )
    if actual_rows != expected_rows:
        raise ValueError(
            f"Expected {expected_rows:,} rows, found {actual_rows:,}"
        )
    if not FULL_RUN:
        check = pl.read_parquet(recommendations_path)
        if check.select(pl.col("cutoff").n_unique()).item() != CANDIDATE_CAP:
            raise ValueError("Smoke output is missing cutoff values")
        if check.group_by("impression_uid").agg(
            pl.col("news_code").n_unique().alias("unique_news")
        ).filter(pl.col("unique_news") != CANDIDATE_CAP).height:
            raise ValueError("Smoke output contains duplicate candidates")
    print(f"[{trial}] Wrote {actual_rows:,} candidate rows in {elapsed:.1f}s")
    subset_uids = (
        np.concatenate(processed_uids)
        if processed_uids
        else np.empty(0, dtype=np.int64)
    )
    return (
        recommendations_path,
        processed_requests,
        actual_rows,
        elapsed,
        subset_uids,
    )

In [ ]:
def smoke_recall(recommendations_path, model_name, impression_uids):
    uid_frame = pl.DataFrame(
        {"impression_uid": np.asarray(impression_uids, dtype=np.int64)}
    )
    label_parts = sorted(
        (FEATURES / f"labels_{TARGET_SPLIT}").glob("*.parquet")
    )
    if not label_parts:
        raise FileNotFoundError(f"No labels for {TARGET_SPLIT}")
    labels = pl.read_parquet(
        label_parts,
        columns=["impression_uid", "news_code", "clicked"],
    ).join(uid_frame, on="impression_uid", how="inner")
    positives = labels.filter(pl.col("clicked") == 1).select(
        "impression_uid", "news_code"
    )
    relevant = positives.group_by("impression_uid").agg(
        pl.len().alias("relevant")
    )
    base = uid_frame.join(
        relevant, on="impression_uid", how="left"
    ).with_columns(pl.col("relevant").fill_null(0).cast(pl.Int64))
    candidates = pl.read_parquet(
        recommendations_path,
        columns=["impression_uid", "news_code", "cutoff"],
    )
    rows = []
    for m in M_VALUES:
        selected = candidates.filter(pl.col("cutoff") <= m)
        hits = (
            positives.join(
                selected.select("impression_uid", "news_code"),
                on=["impression_uid", "news_code"],
                how="inner",
            )
            .group_by("impression_uid")
            .agg(pl.len().alias("hits"))
        )
        per_impression = base.join(
            hits, on="impression_uid", how="left"
        ).with_columns(pl.col("hits").fill_null(0).cast(pl.Int64))
        relevant_clicks = int(per_impression["relevant"].sum())
        retrieved_relevant = int(per_impression["hits"].sum())
        with_relevant = per_impression.filter(pl.col("relevant") > 0)
        recall_impression_mean = (
            float(
                (
                    with_relevant["hits"] / with_relevant["relevant"]
                ).mean()
            )
            if with_relevant.height
            else float("nan")
        )
        hit_rate = (
            float((per_impression["hits"] > 0).sum() / per_impression.height)
            if per_impression.height
            else float("nan")
        )
        rows.append(
            {
                "model": model_name,
                "split": TARGET_SPLIT,
                "m": int(m),
                "recall_pooled": (
                    retrieved_relevant / relevant_clicks
                    if relevant_clicks
                    else float("nan")
                ),
                "recall_impression_mean": recall_impression_mean,
                "hit_rate": hit_rate,
                "impressions": int(per_impression.height),
                "relevant_clicks": relevant_clicks,
                "retrieved_relevant": retrieved_relevant,
                "mean_candidates": float(min(m, CANDIDATE_CAP)),
            }
        )
    return pl.DataFrame(rows)

In [ ]:
for trial in TRIALS:
    state = TRAINED_TRIALS[trial]
    (
        recommendations_path,
        processed_requests,
        actual_rows,
        score_seconds,
        subset_uids,
    ) = score_target_split(trial, state["model"], state["artifact_dir"])
    smoke_recall_path = None
    if not FULL_RUN:
        smoke_evaluation = smoke_recall(
            recommendations_path, state["model_name"], subset_uids
        )
        smoke_recall_path = state["artifact_dir"] / f"smoke_recall_{TARGET_SPLIT}.csv"
        smoke_evaluation.write_csv(smoke_recall_path)
        print(smoke_evaluation.sort("m"))
        print(f"[{trial}] Smoke recall:", smoke_recall_path)
    state.update(
        {
            "recommendations_path": recommendations_path,
            "processed_requests": processed_requests,
            "actual_rows": actual_rows,
            "score_seconds": score_seconds,
            "subset_uids": subset_uids,
            "smoke_recall_path": smoke_recall_path,
        }
    )

In [ ]:
for trial in TRIALS:
    state = TRAINED_TRIALS[trial]
    model_name = state["model_name"]
    artifact_dir = state["artifact_dir"]
    recommendations_path = state["recommendations_path"]
    smoke_recall_path = state["smoke_recall_path"]
    processed_requests = state["processed_requests"]
    actual_rows = state["actual_rows"]
    score_seconds = state["score_seconds"]
    training_log = state["training_log"]
    stopper = state["stopper"]
    manifest = {
        "model": model_name,
        "fusion": trial,
        "score": (
            f"{LOSS.upper()}-trained MLP score from request-level user "
            "features and article-level content/popularity features"
        ),
        "embedding_blocks": ["user_embedding", "item_embedding"],
        "user_features": {
            "selected_interest_categories": {
                "slots": CATEGORY_SLOTS,
                "one_hot_dim": CATEGORY_DIM,
                "padding": "zero vector",
                "encoder": "sklearn OneHotEncoder",
            },
            "history_pool": "L2-normalised mean of valid text64 + entity100",
            "history_source": "full_history_rows",
        },
        "item_features": {
            "category_one_hot_dim": CATEGORY_DIM,
            "text_dim": TEXT_DIM,
            "popularity_dim": POPULARITY_DIM,
        },
        "popularity_features": list(POPULARITY_FEATURES),
        "popularity_weight": POPULARITY_WEIGHT,
        "popularity_prior": "standardized log1p prior exposures",
        "popularity_prior_mean": POPULARITY_PRIOR_MEAN,
        "popularity_prior_std": POPULARITY_PRIOR_STD,
        "popularity_smoothing": POPULARITY_SMOOTHING,
        "popularity_source": "fit-split news_catalogue aggregates",
        "train_protocol": (
            f"request-level {LOSS.upper()} from training_negative_groups; "
            "one sampled observed negative per positive"
        ),
        "train_groups": int(n_groups),
        "max_observed_negatives": int(NEG_MAX),
        "max_epochs": NUM_EPOCHS,
        "best_epoch": int(stopper.best_epoch),
        "epochs": int(stopper.best_epoch),
        "tune_ndcg": float(stopper.best_score),
        "ndcg_k": int(TUNE_NDCG_K),
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
        "fit_split": FIT_SPLIT,
        "target_split": TARGET_SPLIT,
        "candidate_cap": CANDIDATE_CAP,
        "candidate_format": ["impression_uid", "cutoff", "news_code"],
        "eligibility": "available_from <= timestamp",
        "history_policy": "exclude every article in full_history_rows",
        "fallback": "popularity order for empty-history requests or short lists",
        "mode": "full" if FULL_RUN else "smoke",
        "device": DEVICE,
        "torch_version": torch.__version__,
        "user_extra_dim": USER_EXTRA_DIM,
        "item_extra_dim": ITEM_EXTRA_DIM,
        "target_requests": int(processed_requests),
        "recommendation_rows": int(actual_rows),
        "recommendations_file": recommendations_path.name,
        "smoke_recall_file": (
            smoke_recall_path.name if smoke_recall_path is not None else None
        ),
        "score_seconds": float(score_seconds),
        "training_seconds": float(sum(row["seconds"] for row in training_log)),
        "model_file": "model.pt",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    manifest_path = artifact_dir / f"manifest_{TARGET_SPLIT}.json"
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    print(f"[{trial}] Manifest:", manifest_path)

In [ ]:
if FULL_RUN:
    from cg_evaluation import CandidateEvaluator

    for trial in TRIALS:
        state = TRAINED_TRIALS[trial]
        model_name = state["model_name"]
        recommendations_path = state["recommendations_path"]
        evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
        evaluation = evaluator.evaluate(
            recommendations_path,
            model_name=model_name,
            m_values=M_VALUES,
        )
        result_path = (
            RESULTS / f"candidate_generation_{model_name}_{TARGET_SPLIT}.csv"
        )
        evaluation.write_csv(result_path)
        if COMPARISON_PATH.exists():
            existing = pl.read_csv(COMPARISON_PATH).filter(
                pl.col("model") != model_name
            )
            common = [
                column
                for column in existing.columns
                if column in evaluation.columns
            ]
            comparison = pl.concat(
                [existing.select(common), evaluation.select(common)],
                how="vertical_relaxed",
            )
        else:
            comparison = evaluation
        comparison.write_csv(COMPARISON_PATH)
        print(evaluation.sort("m"))
        print(f"[{trial}] Results:", result_path)

    comparison = pl.read_csv(COMPARISON_PATH)
    selected_models = [f"ncf_features_{trial}" for trial in TRIALS]
    print(
        comparison.filter(pl.col("model").is_in(selected_models)).sort(
            ["model", "m"]
        )
    )
else:
    print("Smoke run: shared evaluator skipped; subset recall written.")

## Notes and limitations

- User-side features are request-specific, while user and article ID
  embeddings are shared across requests.
- BPR negatives are observed slate non-clicks, not uniform catalogue samples.
  Scoring the full eligible catalogue is therefore an extrapolation.
- The user history pooling uses the full causal history from
  `full_history_rows`, including repeated references. It does not use any later
  feedback.
- The three interest slots are ordered as supplied; missing slots are zero.
- Popularity features are frozen fit-split catalogue aggregates, not
  per-request causal counts.
- The additive popularity prior is frozen on the fit split and applied only at
  scoring time, so the trained MLP cannot cancel it. `POPULARITY_WEIGHT=0`
  recovers the pure MLP score; larger values force the ranking toward prior
  popularity.
- Empty-history requests fall back to popularity order for both trials.
- Early fusion precomputes the static item tower at scoring time. Late fusion
  must evaluate the pair-level ID-fusion layers for every candidate pair and is
  correspondingly slower.
- The BPR score is a ranking score, not a calibrated click probability.
- Smoke-mode recall covers only the scored request subset and is a pipeline
  check, not a full-split benchmark.
- `available_from` is the project's observable-availability proxy, not a
  verified publication timestamp.